# Consumer Complaint Classification | PyTorch MLP

**TF-IDF tabanlı, dört sınıflı tüketici şikâyeti sınıflandırması**  
**Notebook:** `03_pytorch_mlp.ipynb` · **Görev:** Metinden finansal ürün kategorisini tahmin etme

> **Deney özeti:** 5.000 metin · 4 sınıf · TF-IDF (`max_features=1500`) · MLP (128 / 256 gizli nöron) · Adam (`lr=0.001`) · 10 epoch. 128 nöronlu model **%67,30 accuracy / 0,6739 Macro-F1**, 256 nöronlu model **0,6730 Macro-F1** elde etti. Aynı proje kapsamında Logistic Regression'ın 1.500 özellikli baseline sonucu **0,7197 Macro-F1** idi.

### Bu çalışmada ne yaptım?

1. Finansal şikâyet metinlerini temizleyip dört kategoriye eşledim.
2. Eğitim verisinde TF-IDF sözlüğü oluşturdum; ayrılmış veriye yalnızca `transform` uyguladım.
3. Sparse TF-IDF matrislerini PyTorch için `float32` tensörlere, etiketleri `torch.long` tipine dönüştürdüm.
4. `nn.Module`, Linear, ReLU, `CrossEntropyLoss` ve Adam ile MLP tasarladım.
5. `TensorDataset` / `DataLoader` üzerinden mini-batch eğitim döngüsü yazdım.
6. Accuracy, Macro-F1, classification report ve confusion matrix ile değerlendirdim.
7. 128 → 256 nöron deneyini ve Logistic Regression baseline'ını karşılaştırdım.

**Değerlendirme protokolü notu:** Orijinal kodda `X_test` değişkeni `X_val_tensor` / `val_loader` adına aktarılmıştır. Gerçekte **ayrı validation kümesi oluşturulmamıştır**; mevcut sayılar, eğitim dışında tutulan **aynı %20'lik holdout/test bölümündeki** gözlemlerdir. Bu bölüm iki modelin karşılaştırılmasında kullanıldığı için bağımsız bir nihai test skoru olarak görülmemelidir. Daha güvenilir seçim için train/validation/test ayrımı gerekir.

**Yeniden çalıştırma:** Notebook veriyi dışarıdan `complaints_train_5000.csv` olarak bekler; dosya bu pakette yer almaz. Örnek çıktılar geçmiş koşularda kaydedilmiştir; seed/ortam değişimlerinde rakamlar farklılaşabilir. Kurulum ayrıntıları için [README.md](README.md), kapsamlı yorumlar için [deney raporu](pytorch_experiment_report.md).


## 1 · Kütüphaneler ve çalışma ortamı

PyTorch ile sinir ağı/eğitim, scikit-learn ile ayırma-vektörleştirme-metrikler, pandas ile tablo işlemleri kullanılır. CUDA varsa GPU, yoksa CPU seçilir. `torch.manual_seed(42)` tekrarlanabilirliği artırır; CUDA ve farklı kütüphane sürümleri nedeniyle sonuçları mutlak olarak sabitlemez.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Model parametrelerinin başlangıcı için seed (tam determinism garanti etmez).
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 2 · Veri setini yükleme

Kaynak dosya **`complaints_train_5000.csv`**; ana alanlar `Consumer complaint narrative` (metin) ve `Product` (hedef sınıf). `Complaint ID` model özelliği olarak kullanılmıyor. Verinin dağıtım/erişim koşullarını kontrol etmeden ham şikâyetleri depoya eklememek gerekir.

In [ ]:
# Notebook'u çalıştırdığınız dizine göre veri dosyasını arar.
# CSV bu paketle dağıtılmıyor; README'deki talimata göre yerleştirin.
locations = [
    Path("complaints_train_5000.csv"),
    Path("../complaints_train_5000.csv"),
    Path("data/complaints_train_5000.csv"),
]
data_path = next((path for path in locations if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError(
        "complaints_train_5000.csv bulunamadı. CSV dosyasını notebook ile "
        "aynı dizine, üst dizine veya data/ klasörüne koyun."
    )

df = pd.read_csv(data_path)
df.head()

### 2.1 · Etiketlerin tanımlanması

| Etiket | Finansal ürün kategorisi |
|---:|---|
| `0` | Credit reporting, credit repair services, or other personal consumer reports |
| `1` | Debt collection |
| `2` | Credit reporting |
| `3` | Mortgage |

Son katmandaki nöron sayısı **4**, çünkü hedef sınıflar `0,1,2,3`. Baseline raporunda aynı sınıflar **1–4** olarak kodlanmış olabilir; iki çalışmayı sayıdan değil **kategori adından** karşılaştırmak gerekir.

In [3]:
label_map = {
    "Credit reporting, credit repair services, or other personal consumer reports": 0,
    "Debt collection": 1,
    "Credit reporting": 2,
    "Mortgage":3 
}

df["label"] = df["Product"].map(label_map)

### 2.2 · Metin ön işleme

Eksik metinler boş dizgeye çevrilir; metin küçültülür, maskelenmiş `XXXX` alanları ve noktalama kaldırılır, fazla boşluklar sadeleştirilir. **Not:** Bu basit temizlik bazı bağlam ipuçlarını da silebilir; stopword silme veya kök bulma burada uygulanmadı.

In [4]:
import re
def clean_text(text):

    if pd.isna(text):
        return ""

    # Küçük harfe dönüştür
    text = str(text).lower()

    # XXXX gibi gizlenmiş bilgileri kaldır
    text = re.sub(r"\bx{2,}\b", " ", text)

    # Noktalama işaretlerini kaldır
    text = re.sub(r"[^\w\s]", " ", text)

    # Fazla boşlukları temizle
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [5]:
df["Consumer complaint narrative"] = df["Consumer complaint narrative"].apply(clean_text)

### 2.3 · Özellik ve hedef ayrımı

`X`: temizlenmiş şikâyet metni, `y`: sayısal kategori etiketi. Bu aşamada henüz vektörleştirme yok.

In [6]:
X = df["Consumer complaint narrative"]
y = df["label"]

In [7]:
print("X shape :" ,X.shape)
print("Y shape :" , y.shape)

print(X.head(5))
print(y.head(5))

X shape : (5000,)
Y shape : (5000,)
0    in my was repossessed i recently pulled my cre...
1    of representing a payday loan company spotloan...
2    i have paid my son s all bills at medicine or ...
3    i did not allow this company to do a hard cred...
4    numerous entries on credit report for the same...
Name: Consumer complaint narrative, dtype: str
0    0
1    1
2    1
3    2
4    2
Name: label, dtype: int64


## 3 · Eğitim / holdout ayrımı

`train_test_split(test_size=0.2, random_state=42, stratify=y)` ile **4.000 eğitim** ve **1.000 holdout** örneği oluşturuldu. `stratify=y` sınıf oranlarını korur; holdout bölümünde her sınıftan **250 örnek** vardır. Eğitim dışında tutulan bölüm kodun sonraki hücrelerinde `val` olarak adlandırılsa da ayrı bir validation split değildir.

In [61]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42 , stratify=y ) #Veriyi %80 training ve %20 validation olarak ayırır.

In [62]:
print("Train :" ,  X_train.shape )
print("Test :" , X_test.shape )

Train : (4000,)
Test : (1000,)


## 4 · Metinleri TF-IDF özelliklerine dönüştürme

`max_features=1500`: en fazla 1.500 kelime özelliği. Vectorizer sadece eğitim verisinde `fit_transform`, holdout bölümünde yalnızca `transform` kullanır; bu, sözlük/IDF kaynaklı veri sızıntısını önler.

In [63]:
tf_idf_word_vectorizer  = TfidfVectorizer(max_features=1500) #TF-IDF ile metinleri sayısallaştırır.

X_train_tfidf = tf_idf_word_vectorizer.fit_transform(X_train)
X_test_tfidf = tf_idf_word_vectorizer.transform(X_test)


### 4.1 · Tensor hazırlama

TF-IDF matrisi **SciPy sparse** formattadır; burada basit MLP eğitimi için dense `float32` tensöre dönüştürülür. `y` etiketleri sınıf indeksleri olduğu için **`torch.long`** kullanılır. Eğitim tensörü `(4000, 1500)`, holdout tensörü `(1000, 1500)` boyutundadır.

In [ ]:
# TF-IDF'nin sparse çıktısını yoğun NumPy dizisine, oradan float32 Tensor'a çevir.
X_train_tensor = torch.from_numpy(X_train_tfidf.toarray()).float()

# İsim 'val' olsa da bu tensor, X_test üzerinden oluşturulan holdout bölümüdür.
X_val_tensor = torch.from_numpy(X_test_tfidf.toarray()).float()

In [65]:
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.long) #y_train serisini önce numpy a daha sonra da tensor a çevirdik.
y_val_tensor = torch.tensor(y_test.to_numpy() , dtype=torch.long)  #y_test serisini önce numpy a daha sonra da tensor a çevirdik.

In [66]:
print(X_train_tensor.shape)
print(X_train_tensor.dtype)

print(X_val_tensor.shape)
print(X_val_tensor.dtype)

torch.Size([4000, 1500])
torch.float32
torch.Size([1000, 1500])
torch.float32


### 4.2 · Neden sparse? Bellek maliyeti

Metinler 1.500 kelimelik sözlüğün yalnızca küçük bölümünü içerdiğinden TF-IDF matrisindeki değerlerin çoğu `0` olur. Sparse CSR biçimi yalnızca sıfır olmayan değerleri ve konumlarını saklayarak bellek tasarrufu sağlar.

**Dense `float32` bellek hesabı:** `örnek × özellik × 4 byte`.

| Kayıt | Özellik | Yaklaşık dense bellek |
|---:|---:|---:|
| 5.000 | 1.500 | **30 MB** |
| 1.000.000 | 1.500 | **6 GB** |

Bu, yalnızca yoğun matrisin ham verisidir; dönüştürme sırasında ek belleğe ve eğitim için ara aktivasyonlara ihtiyaç olabilir. Büyük veri için sparse destekli modeller veya batch düzeyinde dönüşüm tercih edilebilir.

## 5 · Model mimarisi: `ComplaintClassifier`

```text
TF-IDF (1500 özellik)
    ↓
Linear(1500 → hidden_dim)
    ↓ ReLU
Linear(hidden_dim → 32)
    ↓ ReLU
Linear(32 → 4)
    ↓ logits
```

128 nöronlu deney için **1500 → 128 → 32 → 4**; daha sonra yalnızca ilk gizli katmanı **256** nörona çıkarıyorum. `nn.Flatten()` gerekmiyor çünkü TF-IDF girdisi zaten `(batch, features)` biçiminde. Son Linear katmanının ardından Softmax yok.

In [68]:
class ComplaintClassifier(nn.Module):

    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()

        # Katmanları burada oluştur
        self.fc1 = nn.Linear(input_dim , hidden_dim)
        self.relu = nn.ReLU()

        self.fc2 = nn.Linear(hidden_dim , 32)
        self.fc3 = nn.Linear(32 , num_classes)

    def forward(self, x):

        # İleri yayılımı burada gerçekleştir
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)

        return x


model = ComplaintClassifier(
    input_dim=1500,
    hidden_dim=128,
    num_classes=4
).to(device)

In [69]:
print(model)

ComplaintClassifier(
  (fc1): Linear(in_features=1500, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=4, bias=True)
)


### 5.1 · Tasarım kararlarını neden böyle verdim?

**ReLU:** Doğrusal olmayan ilişkileri modelleyebilmeyi sağlar; negatif girişleri sıfırlar, pozitifleri korur. İki doğrusal katman arasına aktivasyon konmazsa bileşim tek doğrusal dönüşüme indirgenebilir.

**Dört çıktı:** `0–3` arasında dört kategori bulunduğu için son Linear katman dört **logit** üretir.

**Softmax neden yok?** `nn.CrossEntropyLoss` ham logits bekler ve `log_softmax` ile negatif log-olasılık hesabını kendi içinde yapar. Modele önceden Softmax koymak hesaplanan kaybı değiştirir.

## 6 · Kayıp fonksiyonu ve optimizer

- **`nn.CrossEntropyLoss()`** dört sınıflı sınıflandırmada ham logits ile `torch.long` etiketleri karşılaştırır; Softmax/LogSoftmax işlemini kayıp hesabında kararlı biçimde içerir.
- **`torch.optim.Adam(lr=0.001)`** her parametre için adaptif adım büyüklükleri kullanır; başlangıç için pratik bir tercih olmakla beraber burada diğer optimizerlarla deneysel kıyas yapılmadı.
- `model.parameters()` öğrenilebilir ağırlıkları ve bias değerlerini optimizer’a iletir.

In [71]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

### 6.1 · Parametre sayısını doğrulama

| Katman | Ağırlık + bias | Parametre |
|---|---|---:|
| Linear(1500,128) | `1500×128 + 128` | 192.128 |
| Linear(128,32) | `128×32 + 32` | 4.128 |
| Linear(32,4) | `32×4 + 4` | 132 |
| **Toplam** | | **196.388** |

ReLU’nun öğrenilebilir parametresi yoktur; yalnızca `requires_grad=True` parametreleri sayıldığında da sonuç aynıdır.

In [72]:
total_params = sum(param.numel() for param in model.parameters())

print("Toplam parametre:", total_params)

Toplam parametre: 196388


### 6.2 · Mini-batch ile ileri yayılım ve loss testi

32 rastgele örnekle girdi `(32,1500)` → çıktı `(32,4)` beklenir. Bu testte `model(X_batch)` logits üretir, `criterion(outputs, y_batch)` tek bir loss değeri döndürür. **Bu hücre tek başına eğitim yapmaz**; `backward()` ve `optimizer.step()` kullanılmaz.

In [ ]:
indices = torch.randperm(len(X_train_tensor))[:32]

X_batch = X_train_tensor[indices]
y_batch = y_train_tensor[indices]

# CPU ve GPU'da çalışması için model ile aynı cihazı kullan.
batch_device = next(model.parameters()).device
X_batch = X_batch.to(batch_device)
y_batch = y_batch.to(batch_device)

outputs = model(X_batch)
print(outputs.shape)

loss = criterion(outputs, y_batch)
print("Loss:", loss.item())

In [74]:
trainable_params = sum(
    param.numel()
    for param in model.parameters()
    if param.requires_grad
)

print("Eğitilebilir parametre:", trainable_params)

Eğitilebilir parametre: 196388


## 7 · Eğitim verisi ve mini-batch mantığı

`TensorDataset(X_train_tensor, y_train_tensor)` özellik-etiket eşleşmesini tutar. `DataLoader(batch_size=32, shuffle=True)` her epoch’ta verileri karıştırarak minibatch sunar. **4.000 / 32 = 125 batch/epoch** ve **10 epoch = 1.250 optimizer adımı** (`optimizer.step` her batch’te çağrılır).

In [75]:
dataset = TensorDataset(X_train_tensor , y_train_tensor)

dataloader = DataLoader(dataset=dataset , batch_size=32 , shuffle=True)

len(dataloader)

125

### 7.1 · Training loop: forward → loss → backward → update

Her batch’te: `(1)` tensörleri modelin cihazına taşı, `(2)` önceki gradyanları sıfırla, `(3)` forward ile logits üret, `(4)` CrossEntropyLoss hesapla, `(5)` `loss.backward()` ile gradyanları bul, `(6)` `optimizer.step()` ile ağırlıkları güncelle.

Her epoch sonunda batch kayıplarının ortalaması `train_losses` listesine eklenir. **Bu veri kümesinde batch’lerin hepsi 32 örnek** olduğundan aritmetik ortalama örnek-ağırlıklı ortalamaya eşittir.

In [76]:
def train_model(model, train_loader, criterion, optimizer, epochs=10):

    train_losses = []
    device = next(model.parameters()).device

    model.train()

    for epoch in range(epochs):
        running_loss = 0

        for X_batch, y_batch in train_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)

        train_losses.append(avg_loss)

        print(f"Epoch {epoch + 1}/{epochs} | Loss: {avg_loss:.4f}")

    return train_losses

train_losses = train_model(
    model,
    dataloader,
    criterion,
    optimizer,
    epochs=10
)


Epoch 1/10 | Loss: 1.0802
Epoch 2/10 | Loss: 0.5873
Epoch 3/10 | Loss: 0.4785
Epoch 4/10 | Loss: 0.3974
Epoch 5/10 | Loss: 0.3299
Epoch 6/10 | Loss: 0.2761
Epoch 7/10 | Loss: 0.2300
Epoch 8/10 | Loss: 0.1905
Epoch 9/10 | Loss: 0.1560
Epoch 10/10 | Loss: 0.1226


### 7.2 · Epoch ve batch farkı

- **Batch:** Eğitim verisinin tek optimizasyon adımında kullanılan küçük parçası (burada 32 metin).
- **Epoch:** Tüm 4.000 eğitim örneğinin bir kez işlenmesi (burada 125 batch).

Bu training loop'ta her batch sonunda optimizer bir kez güncellenir. Genel olarak *gradient accumulation* gibi yöntemlerde bir güncelleme birkaç batch sonrasında da yapılabilir.

## 8 · Holdout üzerinde değerlendirme

Bu bölümde kodda `val_loader` adı geçiyor, fakat kullanılan veri ilk aşamada oluşturulan **`X_test / y_test`** bölümüdür. Karşılaştırma/tuning için kullanıldığı için burada **holdout skoru** diyoruz.

`shuffle=False` sıralamayı korur; `model.eval()` değerlendirme modunu açar, `torch.no_grad()` gereksiz gradyan takibini kapatır. `argmax(dim=1)` her örneğin dört logits değeri arasından en yüksek skorlu sınıfı seçer.

In [78]:
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=32,
    shuffle=False
)

print(X_val_tensor.shape)
print(y_val_tensor.shape)
print(len(val_loader))


torch.Size([1000, 1500])
torch.Size([1000])
32


In [79]:
def test_model(model, test_loader):

    model.eval()

    predicted_classes = []
    true_classes = []

    device = next(model.parameters()).device

    with torch.no_grad():

        for X_batch, y_batch in test_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            # Forward Pass
            predictions = model(X_batch)

            # Tahmin edilen sınıfları bul
            predicted = predictions.argmax(dim=1)

            # Tahminleri listeye ekle
            predicted_classes.append(predicted)

            # Gerçek etiketleri listeye ekle
            true_classes.append(y_batch)

    # Bütün batch'leri birleştir
    predicted_classes = torch.cat(predicted_classes)
    true_classes = torch.cat(true_classes)

    # CPU ve NumPy dönüşümü
    predicted_classes = predicted_classes.cpu().numpy()
    true_classes = true_classes.cpu().numpy()

    return true_classes, predicted_classes

y_true, y_pred = test_model(model, val_loader)

print("Gerçek etiketler:", y_true[:10])
print("Tahminler:", y_pred[:10])

print("Gerçek etiket shape:", y_true.shape)
print("Tahmin shape:", y_pred.shape)

Gerçek etiketler: [1 2 0 3 0 2 1 2 0 0]
Tahminler: [0 2 1 3 0 2 1 2 1 0]
Gerçek etiket shape: (1000,)
Tahmin shape: (1000,)


### 8.1 · Metrikler

- **Accuracy:** doğru tahmin / toplam metin.
- **Macro-F1:** her sınıfın F1 skoru eşit ağırlıklı ortalanır; azınlık sınıflarını gözden kaçırmamak için önemlidir.
- **Classification report:** precision, recall, F1 ve support.
- **Confusion matrix:** hangi gerçek sınıfın hangi sınıfa karıştığını gösterir.

In [80]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

accuracy = accuracy_score(y_true, y_pred)

macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

print("Accuracy:", accuracy)
print("Macro-F1:", macro_f1)

Accuracy: 0.673
Macro-F1: 0.673871420434641


### 8.2 · Sınıf bazlı hata analizi (128 nöron)

Kaydedilmiş notebook çıktısında **accuracy %67,30** ve **Macro-F1 0,6739**. Her sınıftan 250 örnek vardır.

| Gerçek sınıf | Precision | Recall | F1 | Destek |
|---|---:|---:|---:|---:|
| `0` · Kredi raporu/onarım | 0,52 | 0,54 | **0,53** | 250 |
| `1` · Borç tahsilatı | 0,72 | 0,71 | 0,72 | 250 |
| `2` · Kredi raporlaması | 0,56 | 0,56 | **0,56** | 250 |
| `3` · Mortgage | 0,90 | 0,89 | **0,89** | 250 |

En belirgin karışıklık **sınıf 0 ↔ sınıf 2**: 0→2 yönünde **76**, 2→0 yönünde **79**, toplam **155** hata. Bu iki kategori metinsel ve kavramsal olarak örtüşebilir; bunun gerçek nedeni ancak yanlış tahmin edilen örnekler incelenerek anlaşılabilir.

![128 nöronlu modelin confusion matrix görseli](figures/confusion_matrix_mlp128.png)

In [81]:
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1, 2, 3],
    zero_division=0
))

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1, 2, 3]
)

print(cm)

              precision    recall  f1-score   support

           0       0.52      0.54      0.53       250
           1       0.72      0.71      0.72       250
           2       0.56      0.56      0.56       250
           3       0.90      0.89      0.89       250

    accuracy                           0.67      1000
   macro avg       0.67      0.67      0.67      1000
weighted avg       0.67      0.67      0.67      1000

[[135  28  76  11]
 [ 34 177  30   9]
 [ 79  26 139   6]
 [ 12  14   2 222]]


### 8.3 · `model.train()` ve `model.eval()` neden ayrı?

- `model.train()`: özellikle Dropout / BatchNorm gibi katmanları eğitim davranışına geçirir; **tek başına model eğitmez**.
- `model.eval()`: katmanların değerlendirme davranışını açar; **tek başına gradient hesaplamasını kapatmaz**.
- `torch.no_grad()`: değerlendirmede gereksiz autograd kaydını önler, bellek kullanımı düşer.
- `argmax(dim=1)`: `(batch, 4)` logits tablosundaki her satırın en yüksek skorlu sınıf indeksini döndürür (`(batch,)`).

## 9 · Hiperparametre deneyi: 128 → 256 gizli nöron

Yalnızca ilk gizli katman genişletilir; giriş boyutu 1500, ikinci gizli katman 32, çıktı 4, Adam learning rate `0.001`, eğitim süresi 10 epoch olarak kalır. Yeni model için **ayrı optimizer** başlatılır.

**Deney sınırı:** İlk notebook çalışmasında seed ve DataLoader karıştırmasının her deney için bağımsız olarak aynı koşullarda sıfırlandığı doğrulanmıyor. Bu nedenle F1 farkını kesin olarak yalnızca nöron sayısına bağlamıyorum. Kontrollü tekrar için her deneyde seed ve DataLoader generator’ı yeniden başlatılmalı.

In [82]:
model_256 = ComplaintClassifier(
    input_dim=1500,
    hidden_dim=256,
    num_classes=4
).to(device)



In [84]:
optimizer_model_256 = torch.optim.Adam(
    model_256.parameters(),
    lr=0.001
)


train_losses_256 = train_model(
    model_256,
    dataloader,
    criterion,
    optimizer_model_256,
    epochs=10
)

Epoch 1/10 | Loss: 1.0498
Epoch 2/10 | Loss: 0.5717
Epoch 3/10 | Loss: 0.4420
Epoch 4/10 | Loss: 0.3458
Epoch 5/10 | Loss: 0.2674
Epoch 6/10 | Loss: 0.2080
Epoch 7/10 | Loss: 0.1512
Epoch 8/10 | Loss: 0.1101
Epoch 9/10 | Loss: 0.0687
Epoch 10/10 | Loss: 0.0406


In [85]:
y_true_256, y_pred_256 = test_model(
    model_256,
    val_loader
)

### 9.1 · Kaydedilmiş karşılaştırma

| Metrik | MLP · 128 nöron | MLP · 256 nöron |
|---|---:|---:|
| Parametre sayısı | 196.388 | 392.612 |
| İlk training loss | 1,0802 | 1,0498 |
| 10. epoch training loss | 0,1226 | **0,0406** |
| Holdout Macro-F1 | **0,6739** | 0,6730 |
| Holdout accuracy | **%67,30** | Notebook’ta ayrıca ölçülmedi |

256 nöronlu model eğitim kümesine daha iyi uyum sağlamasına rağmen holdout Macro-F1 iyileşmemiş. Bu durum **olası overfitting / genelleme sınırı** ile uyumlu olabilir; epoch bazlı validation loss ve kontrollü tekrar olmadan kesin teşhis konulamaz.

![128 ve 256 nöron eğitim loss eğrileri](figures/training_loss_comparison.png)

In [86]:
macro_f1_256 = f1_score(
    y_true_256,
    y_pred_256,
    average="macro"
)

print("Macro-F1:", macro_f1_256)

Macro-F1: 0.6729960801458406


## 10 · Logistic Regression ile karşılaştırma

Önceki `baseline_report.md` içindeki Logistic Regression sonuçlarını, bu notebook'taki *kaydedilmiş* holdout metrikleriyle karşılaştırıyorum. Baseline çalışmasında sınıf numaraları `1–4`, bu notebook'ta `0–3` olsa da **kategori adları aynıdır**.

| Yöntem | TF-IDF özellik sayısı | Accuracy | Macro-F1 |
|---|---:|---:|---:|
| Logistic Regression | 1.500 | **%72,30** | **0,7197** |
| Logistic Regression | 3.000 | **%72,60** | **0,7231** |
| PyTorch MLP · 128 nöron | 1.500 | %67,30 | 0,6739 |
| PyTorch MLP · 256 nöron | 1.500 | — | 0,6730 |

**Sonuç:** Bu veri/özellik ayarlarında sparse TF-IDF kullanan **Logistic Regression daha iyi Macro-F1** sağlıyor. İlk 1.500 özellikli baseline ile 128 nöronlu MLP arasındaki fark yaklaşık **4,58 yüzde puan**. Daha karmaşık bir model her zaman daha iyi genellemez.

## 11 · Ne öğrendim? Sonraki adımlar

**Öğrendiklerim:** (i) sparse TF-IDF → dense float32 Tensor dönüşümü ve bellek maliyeti, (ii) `nn.Linear`/ReLU/logits/parametre hesabı, (iii) `TensorDataset` ve `DataLoader` ile mini-batch, (iv) `zero_grad → forward → loss → backward → step`, (v) `eval + no_grad + argmax`, (vi) sınıf bazlı değerlendirme ve baseline'ın önemi.

**Bir sonraki kontrollü deney:** Ayrı **train/validation/test** bölünmesi; her konfigürasyonun aynı seed ve karıştırma koşullarıyla sıfırdan eğitilmesi; her epoch'ta validation loss/F1 izleme ve **early stopping**. Sonra ReLU ağında Dropout/weight decay, TF-IDF `max_features` ve `ngram_range` denemeleri. Nihai test kümesini model seçimi tamamlanana kadar saklayacağım.

**Temel sınırlama:** Eğitim kümesine düşen kayıp, genelleme kalitesinin tek başına ölçüsü değildir; burada yalnızca kayıtlı holdout metrikleri mevcut, tekrarlar/güven aralıkları yok.

→ Ayrıntılı dokümantasyon: [README](README.md) · [Deney raporu](pytorch_experiment_report.md)